<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Wan2.2_S2V_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Wan 2.2 S2V — Audio-Driven Cinematic Video (Single-GPU Optimized + Cloud API)

A self-contained Colab wrapper around [Tencent / Wan-AI's Wan 2.2 S2V-14B](https://huggingface.co/Wan-AI/Wan2.2-S2V-14B) — the **14 B MoE** audio-driven video generation model. Takes a **character image** + **audio clip** and produces a cinematic video of the character speaking, singing, or reacting to the audio.

- **Two execution paths in one notebook**:
  - **💻 Local Single-GPU Engine (Upgraded — fits 24 GB L4 / 40 GB A100)** — clones [Wan-Video/Wan2.2](https://github.com/Wan-Video/Wan2.2), downloads `Wan-AI/Wan2.2-S2V-14B`, and runs `wan.WanS2V` with **CPU initialization + `t5_cpu=True` + `offload_model=True` + `torchao` INT8 DiT quantization** (shrinking active DiT weights from 28 GB to ~14 GB) plus optional **4-step Lightning LoRA** (`lightx2v/Wan2.2-Distill-Loras`) for **10× faster** local generation.
  - **🌐 Cloud API** — uses Alibaba's DashScope API (`wan2.2-s2v`). No GPU needed, ~\$0.10–0.50/call, works on any Colab runtime.
- **Two inputs**: character image + audio clip (`.wav` or `.mp3`)
- **Optional**: text prompt (describes the scene), pose video (drives body motion)
- **Two resolution levels**: `480P` (`480×832`, recommended for 24 GB L4) and `720P` (`720×1280`, recommended for 40 GB A100)
- **Apache 2.0 license**
- **Drive-cached outputs** — generated videos mirror to `/content/drive/MyDrive/AEI_3D_Out/Wan2.2-S2V/`

### Quick start (Local Single-GPU path — L4 24 GB / A100 40 GB)

1. Pick an **L4 (24 GB)** or **A100 (40 GB)** runtime and run Steps 1–4.
2. Open the **💻 Local (Single-GPU S2V-14B)** tab in the Gradio UI.
3. Keep **INT8 Quantization (`torchao`)** and **4-Step Lightning LoRA (`lightx2v`)** enabled.
4. Upload a character image + audio file and click **Generate (Local)**.

### Quick start (Cloud API path)

1. **Sign up for Alibaba Cloud's DashScope** at [dashscope.aliyun.com](https://dashscope.aliyun.com/) and activate the **wan2.2-s2v** service.
2. **Add your API key** to Colab secrets (left sidebar → 🔑) as `DASHSCOPE_API_KEY`.
3. Run Steps 1–4, open the **🌐 Cloud API** tab, upload a character image + audio clip, and click **Generate (Cloud API)**.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs `gradio==5.49.1`, `requests` (for the Cloud API), and the audio/video + `torchao`
# @markdown dependencies required for single-GPU local `WanS2V` inference.

import os, sys, subprocess, time

t0 = time.time()
print('[pip] Installing core + S2V local & cloud dependencies ...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'requests',
    'gradio==5.49.1',
    'tqdm==4.66.5',
    'huggingface-hub>=0.26.0',
    'accelerate>=1.1.0',
    'transformers>=4.46.0',
    'diffusers>=0.31.0',
    'peft>=0.13.0',
    'safetensors',
    'einops',
    'easydict',
    'ftfy',
    'librosa',
    'soundfile',
    'imageio',
    'imageio-ffmpeg',
    'decord',
    'opencv-python-headless',
    'torchao',
], check=True)
print(f'[Install] Dependencies ready in {time.time() - t0:.1f}s. Run Step 2 next.')

In [ ]:
# @title STEP 2 — Credentials & Cache Setup
# @markdown Mounts Google Drive for Hugging Face weight caching (`HF_HOME`) and output persistence,
# @markdown reads `DASHSCOPE_API_KEY` from Colab secrets (optional, for the Cloud API tab), and
# @markdown sets default local S2V optimization flags.

import os, sys, pathlib, time, json, uuid

DEFAULT_QUANTIZE_INT8 = True  # @param {type:"boolean"}
DEFAULT_FAST_4STEP_LORA = True  # @param {type:"boolean"}

# --- Drive mount + output cache ---
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache/huggingface/hub')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(CACHE_ROOT.parent)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT)
    print(f'[Cache] HF cache root: {CACHE_ROOT}')
    HAS_DRIVE = True
except Exception:
    CACHE_ROOT = pathlib.Path('/content/hf_cache/hub')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(CACHE_ROOT.parent)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT)
    print('[Cache] Drive not available, using local /content/hf_cache only.')
    HAS_DRIVE = False

OUT_DIR = pathlib.Path('/content/wan22_s2v_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
if HAS_DRIVE:
    DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Wan2.2-S2V')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'[Drive] Video cache: {DRIVE_OUT}')
else:
    DRIVE_OUT = OUT_DIR

# --- Read the DashScope API key from Colab secrets ---
DASHSCOPE_API_KEY = ''
try:
    from google.colab import userdata
    DASHSCOPE_API_KEY = userdata.get('DASHSCOPE_API_KEY') or ''
    if DASHSCOPE_API_KEY:
        os.environ['DASHSCOPE_API_KEY'] = DASHSCOPE_API_KEY
        print('[Credentials] DASHSCOPE_API_KEY loaded from Colab secrets.')
    else:
        print('[Credentials] DASHSCOPE_API_KEY is NOT set — Cloud API tab disabled; Local S2V tab works without a key.')
except Exception as e:
    print(f'[Credentials] Could not read Colab secrets ({e}) — Local S2V tab remains available.')

print(f'[Config] Local S2V defaults: INT8={DEFAULT_QUANTIZE_INT8}, Fast 4-Step LoRA={DEFAULT_FAST_4STEP_LORA}')
print('Run Step 3 next.')

In [ ]:
# @title STEP 3 — Client Wrappers (Cloud API + Single-GPU Local S2V Engine)
# @markdown Defines two engines:
# @markdown - `WanS2VCloud`: async-job wrapper for Alibaba's DashScope API (`wan2.2-s2v`).
# @markdown - `WanS2VLocalEngine`: complete single-GPU local inference wrapper around `wan.WanS2V` (`s2v-14B`)
# @markdown   with CPU initialization, `t5_cpu=True`, `convert_model_dtype=True`, `offload_model=True`,
# @markdown   PyTorch SDPA fallback (when `flash_attn` is absent), optional 4-step `lightx2v` Distill LoRA,
# @markdown   and `torchao` INT8 DiT quantization so the 14 B model runs on 24 GB L4 / 40 GB A100 GPUs.

import os, sys, time, gc, json, traceback, shutil, uuid, pathlib, re, subprocess
import requests, tempfile, threading
from typing import Optional
from pathlib import Path
import torch
from IPython.display import display, FileLink

REPO_ID = 'Wan-AI/Wan2.2-S2V-14B'
LOCAL_REPO = 'Wan-Video/Wan2.2'
LOCAL_DIR = pathlib.Path('/content/Wan2.2')
DISTILL_LORA_REPO = 'lightx2v/Wan2.2-Distill-Loras'
DISTILL_LORA_FILE = 'wan2.2_i2v_A14b_low_noise_lora_rank64_lightx2v_4step_1022.safetensors'
CLOUD_URL = 'https://dashscope.aliyuncs.com/api/v1/services/aigc/image2video/video-synthesis/'
CLOUD_POLL = 'https://dashscope.aliyuncs.com/api/v1/tasks/'

RESOLUTIONS = ['480P', '720P']
STYLES = ['speech']
DEFAULT_RESOLUTION = '480P'


# ---------- Cloud API client ----------
class WanS2VCloud:
    """Alibaba DashScope async job. Uploads inputs to OSS, submits, polls, downloads result."""
    def __init__(self, api_key: str = ''):
        self.api_key = api_key or os.environ.get('DASHSCOPE_API_KEY', '')

    def _upload(self, model_id: str, local_path: str) -> str:
        pol_r = requests.get(
            'https://dashscope.aliyuncs.com/api/v1/uploads',
            params={'action': 'getPolicy', 'model': model_id},
            headers={'Authorization': f'Bearer {self.api_key}', 'Content-Type': 'application/json'},
            timeout=60,
        )
        pol_r.raise_for_status()
        policy = pol_r.json()['data']
        key = f"{policy['upload_dir']}/{Path(local_path).name}"
        with open(local_path, 'rb') as f:
            files = {
                'OSSAccessKeyId': (None, policy['oss_access_key_id']),
                'Signature': (None, policy['signature']),
                'policy': (None, policy['policy']),
                'x-oss-object-acl': (None, policy['x_oss_object_acl']),
                'key': (None, key),
                'success_action_status': (None, '200'),
                'file': (Path(local_path).name, f, 'application/octet-stream'),
            }
            r = requests.post(policy['upload_host'], files=files, timeout=300)
        r.raise_for_status()
        return f'oss://{key}'

    def submit(self, image_path: str, audio_path: str, resolution: str = '480P',
               style: str = 'speech', poll_secs: int = 5, max_wait: int = 1800,
               on_progress=None) -> str:
        if not self.api_key:
            raise RuntimeError('DASHSCOPE_API_KEY is not set. Set it in Colab secrets and re-run Step 2.')
        if on_progress: on_progress(0.05, 'Uploading image to OSS ...')
        model_id = 'wan2.2-s2v'
        image_url = self._upload(model_id, image_path)
        if on_progress: on_progress(0.15, 'Uploading audio to OSS ...')
        audio_url = self._upload(model_id, audio_path)
        if on_progress: on_progress(0.25, 'Submitting async job ...')
        payload = {
            'model': model_id,
            'input': {'image_url': image_url, 'audio_url': audio_url},
            'parameters': {'style': style, 'resolution': resolution},
        }
        headers = {
            'X-DashScope-Async': 'enable',
            'X-DashScope-OssResourceResolve': 'enable',
            'Authorization': f'Bearer {self.api_key}',
            'Content-Type': 'application/json',
        }
        r = requests.post(CLOUD_URL, json=payload, headers=headers, timeout=60)
        r.raise_for_status()
        task_id = r.json().get('output', {}).get('task_id')
        if not task_id:
            raise RuntimeError(f'No task_id in response: {r.text}')
        t0 = time.time()
        while True:
            r = requests.get(CLOUD_POLL + task_id,
                              headers={'Authorization': f'Bearer {self.api_key}'}, timeout=60)
            r.raise_for_status()
            out = r.json().get('output', {})
            status = out.get('task_status')
            if status == 'SUCCEEDED':
                if on_progress: on_progress(0.95, 'Downloading result ...')
                video_url = out['results']['video_url']
                out_path = OUT_DIR / f'wan22_s2v_{int(time.time())}_{uuid.uuid4().hex[:6]}.mp4'
                r2 = requests.get(video_url, timeout=300, stream=True)
                r2.raise_for_status()
                with open(out_path, 'wb') as f:
                    for chunk in r2.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
                if on_progress: on_progress(1.0, 'Done.')
                return str(out_path)
            if status in ('PENDING', 'RUNNING'):
                if on_progress:
                    elapsed = int(time.time() - t0)
                    on_progress(min(0.9, 0.25 + 0.6 * elapsed / 600), f'{status} ({elapsed}s)')
                time.sleep(poll_secs)
            else:
                raise RuntimeError(f'Cloud job failed: {out}')
            if time.time() - t0 > max_wait:
                raise TimeoutError(f'Cloud job did not finish within {max_wait}s (task_id={task_id})')


# ---------- Local Single-GPU inference engine ----------
class WanS2VLocalEngine:
    """Single-GPU optimized local loader for Wan 2.2 S2V 14 B (fits 24 GB L4 / 40 GB A100 with INT8 + CPU offload)."""
    def __init__(self):
        self.pipeline = None
        self.quantized = False
        self.fast_lora = False
        self.dtype = torch.bfloat16
        self._repo_dir = None

    def is_ready(self) -> bool:
        return self.pipeline is not None

    def _ensure_repo(self) -> pathlib.Path:
        """Clone Wan-Video/Wan2.2 and apply single-GPU CPU-init + SDPA fallback patches."""
        if not LOCAL_DIR.exists():
            print(f'[Local] Cloning {LOCAL_REPO} into {LOCAL_DIR} ...')
            subprocess.run(['git', 'clone', '--depth=1', f'https://github.com/{LOCAL_REPO}.git', str(LOCAL_DIR)], check=True)
        self._repo_dir = LOCAL_DIR
        if str(LOCAL_DIR) not in sys.path:
            sys.path.insert(0, str(LOCAL_DIR))

        # Patch wan/speech2video.py so init_on_cpu=True loads WanModel_S2V onto CPU first instead of cuda:0
        s2v_py = LOCAL_DIR / 'wan' / 'speech2video.py'
        if s2v_py.exists():
            txt = s2v_py.read_text(encoding='utf-8')
            target = 'device_map=self.device'
            replacement = "device_map=('cpu' if self.init_on_cpu else self.device)"
            if target in txt and replacement not in txt:
                s2v_py.write_text(txt.replace(target, replacement), encoding='utf-8')
                print('[Local] Patched wan/speech2video.py for CPU initialization.')

        # Patch wan.modules.attention.flash_attention to fall back to PyTorch SDPA if flash_attn is not installed
        import wan.modules.attention as wan_attn
        if not getattr(wan_attn, 'FLASH_ATTN_2_AVAILABLE', False) and not getattr(wan_attn, 'FLASH_ATTN_3_AVAILABLE', False):
            _orig_attention = wan_attn.attention
            def _sdpa_fallback(q, k, v, q_lens=None, k_lens=None, dropout_p=0.0, softmax_scale=None, q_scale=None,
                               causal=False, window_size=(-1, -1), deterministic=False, dtype=torch.bfloat16, version=None):
                return _orig_attention(
                    q, k, v, q_lens=q_lens, k_lens=k_lens, dropout_p=dropout_p,
                    softmax_scale=softmax_scale, q_scale=q_scale, causal=causal,
                    window_size=window_size, deterministic=deterministic, dtype=dtype, fa_version=version,
                )
            wan_attn.flash_attention = _sdpa_fallback
            import wan.modules.s2v.model_s2v as model_s2v_mod
            model_s2v_mod.flash_attention = _sdpa_fallback
            print('[Local] Enabled PyTorch SDPA fallback for WanS2V attention.')
        return self._repo_dir

    def _ensure_weights(self) -> pathlib.Path:
        """Download Wan-AI/Wan2.2-S2V-14B weights via huggingface_hub."""
        from huggingface_hub import snapshot_download
        weights_dir = pathlib.Path(CACHE_ROOT) / REPO_ID.replace('/', '_')
        weights_dir.mkdir(parents=True, exist_ok=True)
        if (weights_dir / 'config.json').exists() and (weights_dir / 'Wan2.1_VAE.pth').exists():
            existing = sum(f.stat().st_size for f in weights_dir.rglob('*') if f.is_file())
            if existing > 20 * 1024**3:
                print(f'[Local] Weights already cached at {weights_dir} ({existing / 1024**3:.1f} GB)')
                return weights_dir
        print(f'[Local] Downloading {REPO_ID} (~27 GB) to {weights_dir} ...')
        snapshot_download(repo_id=REPO_ID, local_dir=str(weights_dir), local_dir_use_symlinks=False)
        return weights_dir

    def _merge_fast_lora(self, noise_model, scale: float = 1.0):
        """Merge lightx2v 4-step Distill LoRA weights into matching linear layers of noise_model on CPU."""
        from huggingface_hub import hf_hub_download
        from safetensors.torch import load_file
        print(f'[Local] Downloading 4-step Lightning LoRA ({DISTILL_LORA_FILE}) ...')
        lora_path = hf_hub_download(repo_id=DISTILL_LORA_REPO, filename=DISTILL_LORA_FILE)
        sd = load_file(lora_path, device='cpu')
        param_dict = dict(noise_model.named_parameters())
        merged = 0
        with torch.no_grad():
            for k, v in sd.items():
                if not k.endswith('.lora_down.weight'):
                    continue
                base_prefix = k[:-len('.lora_down.weight')]
                up_key = base_prefix + '.lora_up.weight'
                alpha_key = base_prefix + '.alpha'
                if up_key not in sd:
                    continue
                # Strip common diffusion_model. / base_model.model. prefixes
                clean = re.sub(r'^(base_model\.model\.|diffusion_model\.|transformer\.)', '', base_prefix) + '.weight'
                if clean not in param_dict:
                    continue
                target_p = param_dict[clean]
                down = sd[k].to(dtype=torch.float32)
                up = sd[up_key].to(dtype=torch.float32)
                rank = down.shape[0]
                alpha = float(sd[alpha_key].item()) if alpha_key in sd else float(rank)
                delta = (up @ down) * (scale * alpha / rank)
                if delta.shape == target_p.shape:
                    target_p.add_(delta.to(dtype=target_p.dtype, device=target_p.device))
                    merged += 1
        del sd
        gc.collect()
        print(f'[Local] Merged {merged} 4-step Lightning LoRA tensors into WanS2V DiT.')

    def load(self, quantize_int8: Optional[bool] = None, use_fast_lora: Optional[bool] = None):
        if quantize_int8 is None:
            quantize_int8 = DEFAULT_QUANTIZE_INT8
        if use_fast_lora is None:
            use_fast_lora = DEFAULT_FAST_4STEP_LORA

        if self.pipeline is not None and self.quantized == bool(quantize_int8) and self.fast_lora == bool(use_fast_lora):
            return
        if not torch.cuda.is_available():
            raise RuntimeError('No CUDA GPU detected — Local S2V requires an L4 (24 GB) or A100 (40/80 GB) GPU.')

        self.unload()
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'[Local] Initializing Wan 2.2 S2V-14B on {torch.cuda.get_device_name(0)} ({vram:.1f} GB VRAM) ...')
        if vram < 38 and not quantize_int8:
            print('[Local] Auto-enabling INT8 quantization so S2V-14B fits in <38 GB VRAM.')
            quantize_int8 = True

        self._ensure_repo()
        weights = self._ensure_weights()

        import wan
        from wan.configs import WAN_CONFIGS
        cfg = WAN_CONFIGS['s2v-14B']
        t0 = time.time()
        s2v = wan.WanS2V(
            config=cfg,
            checkpoint_dir=str(weights),
            device_id=0,
            rank=0,
            t5_fsdp=False,
            dit_fsdp=False,
            use_sp=False,
            t5_cpu=True,
            init_on_cpu=True,
            convert_model_dtype=True,
        )
        if use_fast_lora:
            try:
                self._merge_fast_lora(s2v.noise_model)
            except Exception as e:
                print(f'[Local] 4-step LoRA merge skipped ({e}); continuing with standard weights.')
                use_fast_lora = False

        if quantize_int8:
            try:
                from torchao.quantization import quantize_, Int8WeightOnlyConfig
                print('[Local] Quantizing WanModel_S2V DiT blocks to INT8 via torchao (~14 GB footprint) ...')
                quantize_(s2v.noise_model, Int8WeightOnlyConfig())
            except Exception as e:
                print(f'[Local] torchao INT8 quantization skipped ({e}); continuing in bf16.')
                quantize_int8 = False

        self.pipeline = s2v
        self.quantized = bool(quantize_int8)
        self.fast_lora = bool(use_fast_lora)
        print(f'[Local] WanS2V ready in {time.time() - t0:.1f}s (INT8={self.quantized}, 4-Step LoRA={self.fast_lora}).')

    def unload(self):
        if self.pipeline is not None:
            del self.pipeline
            self.pipeline = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print('[Local] Unloaded WanS2V and freed VRAM.')

    def generate(
        self,
        image_path: str,
        audio_path: str,
        prompt: str = '',
        resolution: str = '480P',
        style: str = 'speech',
        pose_path: Optional[str] = None,
        infer_frames: int = 80,
        steps: Optional[int] = None,
        cfg: Optional[float] = None,
        shift: float = 3.0,
        seed: int = 42,
        quantize_int8: Optional[bool] = None,
        use_fast_lora: Optional[bool] = None,
        on_progress=None,
    ) -> str:
        if on_progress:
            on_progress(0.08, 'Loading single-GPU WanS2V engine ...')
        self.load(quantize_int8=quantize_int8, use_fast_lora=use_fast_lora)

        from wan.configs import WAN_CONFIGS, MAX_AREA_CONFIGS
        from wan.utils.utils import save_video, merge_video_audio

        cfg_obj = WAN_CONFIGS['s2v-14B']
        size_key = '720*1280' if str(resolution).upper() == '720P' else '480*832'
        max_area = MAX_AREA_CONFIGS[size_key]
        actual_steps = int(steps if steps is not None else (4 if self.fast_lora else 40))
        actual_cfg = float(cfg if cfg is not None else (1.0 if self.fast_lora else 4.5))
        # infer_frames must be a multiple of 4
        infer_frames = max(16, (int(infer_frames) // 4) * 4)

        if on_progress:
            on_progress(0.20, f'Running WanS2V ({actual_steps} steps @ {resolution}, {infer_frames} frames/clip) ...')

        video = self.pipeline.generate(
            input_prompt=prompt or 'A person speaking naturally with expressive facial movements and clear lip sync.',
            ref_image_path=str(image_path),
            audio_path=str(audio_path),
            enable_tts=False,
            tts_prompt_audio=None,
            tts_prompt_text=None,
            tts_text=None,
            num_repeat=None,
            pose_video=str(pose_path) if pose_path else None,
            max_area=max_area,
            infer_frames=infer_frames,
            shift=float(shift),
            sample_solver='unipc',
            sampling_steps=actual_steps,
            guide_scale=actual_cfg,
            seed=int(seed),
            offload_model=True,
        )

        if on_progress:
            on_progress(0.90, 'Encoding MP4 and muxing audio track ...')
        out_path = OUT_DIR / f'wan22_s2v_local_{int(time.time())}_{uuid.uuid4().hex[:6]}.mp4'
        save_video(
            tensor=video[None],
            save_file=str(out_path),
            fps=cfg_obj.sample_fps,
            nrow=1,
            normalize=True,
            value_range=(-1, 1),
        )
        merge_video_audio(video_path=str(out_path), audio_path=str(audio_path))
        if on_progress:
            on_progress(1.0, 'Done.')
        return str(out_path)


CLOUD = WanS2VCloud(api_key=DASHSCOPE_API_KEY)
LOCAL_ENGINE = WanS2VLocalEngine()
print('[Core] Cloud client ready (DASHSCOPE_API_KEY set:', bool(CLOUD.api_key), ')')
print('[Core] Local single-GPU S2V engine ready (lazy — loads on first Local generation).')

In [ ]:
# @title STEP 4 — Launch Gradio UI
# @markdown Opens a Gradio app with two tabs: **Local Single-GPU S2V-14B** (with `torchao` INT8 + 4-step Lightning LoRA)
# @markdown and **Cloud API** (DashScope `wan2.2-s2v`).

import os, sys, time, json, uuid, traceback, pathlib, shutil
import torch, gradio as gr
from PIL import Image
from IPython.display import clear_output, display, FileLink


def _new_out(suffix='.mp4'):
    p = DRIVE_OUT / f'wan22_s2v_{int(time.time())}_{uuid.uuid4().hex[:6]}{suffix}'
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


def _err(msg):
    return None, None, 'ERROR: ' + msg


# --- Cloud tab ---
def do_cloud(image, audio, prompt, resolution, progress=gr.Progress()):
    if image is None: return _err('Upload a character image first.')
    if audio is None: return _err('Upload an audio clip first (.wav or .mp3).')
    if not CLOUD.api_key:
        return _err('DASHSCOPE_API_KEY is not set. Add it to Colab secrets and re-run Step 2, or use the Local tab.')
    try:
        progress(0.05, desc='Uploading image to OSS ...')
        out_path = CLOUD.submit(
            image_path=image,
            audio_path=audio,
            resolution=resolution,
            on_progress=lambda p, m: progress(min(0.95, p), desc=m),
        )
        final = _new_out()
        shutil.copy2(out_path, str(final))
        progress(1.0, desc='Done.')
        return str(final), str(image), f'Generated (Cloud) → `{final}`'
    except Exception as e:
        traceback.print_exc()
        return _err(str(e))


# --- Local tab ---
def on_fast_lora_toggle(use_fast):
    if use_fast:
        return gr.update(value=4), gr.update(value=1.0)
    return gr.update(value=40), gr.update(value=4.5)


def do_local(image, audio, prompt, resolution, pose_video, infer_frames, steps, cfg, seed, quantize_int8, use_fast_lora, progress=gr.Progress()):
    if image is None: return _err('Upload a character image first.')
    if audio is None: return _err('Upload an audio clip first.')
    try:
        t0 = time.time()
        out_path = LOCAL_ENGINE.generate(
            image_path=image,
            audio_path=audio,
            prompt=prompt,
            resolution=resolution,
            style='speech',
            pose_path=pose_video,
            infer_frames=int(infer_frames),
            steps=int(steps),
            cfg=float(cfg),
            seed=int(seed),
            quantize_int8=bool(quantize_int8),
            use_fast_lora=bool(use_fast_lora),
            on_progress=lambda p, m: progress(min(0.95, p), desc=m),
        )
        final = _new_out()
        shutil.copy2(out_path, str(final))
        progress(1.0, desc='Done.')
        return str(final), str(image), f'Generated (Local S2V) in {time.time() - t0:.1f}s → `{final}`'
    except Exception as e:
        traceback.print_exc()
        return _err(str(e))


def local_load(quantize_int8, use_fast_lora):
    try:
        LOCAL_ENGINE.load(quantize_int8=bool(quantize_int8), use_fast_lora=bool(use_fast_lora))
        return f'Local S2V engine loaded (INT8={LOCAL_ENGINE.quantized}, 4-Step LoRA={LOCAL_ENGINE.fast_lora}).'
    except Exception as e:
        traceback.print_exc()
        return 'Local load failed: ' + str(e)


def vram_status():
    if not torch.cuda.is_available():
        return 'No GPU.'
    a = torch.cuda.memory_allocated() / 1024**3
    r = torch.cuda.memory_reserved() / 1024**3
    t = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return (
        f'Allocated: {a:.1f} GB\n'
        f'Reserved:  {r:.1f} GB\n'
        f'Total:     {t:.1f} GB\n'
        f'Local engine loaded: {LOCAL_ENGINE.is_ready()} (INT8={LOCAL_ENGINE.quantized}, 4-Step={LOCAL_ENGINE.fast_lora})'
    )


with gr.Blocks(title='Wan 2.2 S2V Colab', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Wan 2.2 S2V — Audio-Driven Cinematic Video\nLip-sync and animate a character image to speech or singing audio. Apache 2.0.')

    gr.HTML(
        '<div style="background: #dcfce7; color: #166534; padding: 8px 12px; border-radius: 6px; margin: 6px 0;">'
        '<b>💻 Local Single-GPU Tab:</b> Uses CPU offload + <code>t5_cpu=True</code> + <code>torchao</code> INT8 DiT quantization '
        'and optional <b>4-Step Lightning LoRA</b> so S2V-14B runs on <b>24 GB L4</b> and <b>40 GB A100</b> GPUs.'
        '</div>'
        '<div style="background: #fef3c7; color: #92400e; padding: 8px 12px; border-radius: 6px; margin: 6px 0;">'
        '<b>🌐 Cloud Tab:</b> Works on any Colab runtime (even CPU) with <code>DASHSCOPE_API_KEY</code> (~$0.10–0.50/call).'
        '</div>'
    )

    with gr.Tabs():
        # -- Local tab --
        with gr.Tab('💻 Local (Single-GPU S2V-14B)'):
            with gr.Row():
                with gr.Column(scale=2):
                    img_local = gr.Image(
                        label='Character image (the subject to lip-sync)',
                        type='filepath', image_mode='RGB', height=240,
                    )
                    aud_local = gr.Audio(
                        label='Audio clip (.wav or .mp3)',
                        type='filepath', sources=['upload'],
                        info='WAV or MP3 speech/singing audio. Output video is automatically muxed with this audio.',
                    )
                    prompt_local = gr.Textbox(
                        label='Prompt (optional)',
                        value='A person speaking naturally with expressive facial movements and clear lip sync.',
                        lines=2,
                        info='Describes the character demeanor, camera framing, and scene context.',
                    )
                    pose_local = gr.Video(
                        label='Pose video (optional — for pose-driven generation)',
                        sources=['upload'],
                        info='Optional driving pose video sequence.',
                    )
                    with gr.Row():
                        res_local = gr.Dropdown(
                            choices=RESOLUTIONS, value=DEFAULT_RESOLUTION, label='Resolution',
                            info='480P (480x832) recommended for 24 GB L4; 720P (720x1280) for 40 GB A100.',
                        )
                        frames_local = gr.Slider(
                            minimum=16, maximum=80, step=4, value=48, label='Frames per Clip (multiple of 4)',
                            info='48 frames (~3s @ 16fps) is fast and low-VRAM; 80 frames (~5s @ 16fps) is full clip length.',
                        )
                    with gr.Row():
                        steps_local = gr.Slider(
                            minimum=4, maximum=50, step=1, value=4 if DEFAULT_FAST_4STEP_LORA else 40,
                            label='Sampling Steps',
                            info='4 steps when 4-Step Lightning LoRA is enabled; 40 steps for base model.',
                        )
                        cfg_local = gr.Slider(
                            minimum=1.0, maximum=7.0, step=0.1, value=1.0 if DEFAULT_FAST_4STEP_LORA else 4.5,
                            label='Guidance Scale (CFG)',
                            info='1.0 for 4-Step Lightning LoRA; 4.5 for standard 40-step inference.',
                        )
                    seed_local = gr.Number(
                        value=42, precision=0, label='Seed (-1 = random)',
                        info='Random seed for reproducible generation.',
                    )
                    with gr.Row():
                        fast_lora_local = gr.Checkbox(
                            value=DEFAULT_FAST_4STEP_LORA,
                            label='4-Step Lightning LoRA (lightx2v — 10x faster)',
                            info='Merges lightx2v 4-step distill LoRA so generation finishes in 4 steps at CFG 1.0.',
                        )
                        quant_local = gr.Checkbox(
                            value=DEFAULT_QUANTIZE_INT8,
                            label='INT8 DiT Quantization (torchao — fits 24 GB L4)',
                            info='Compresses WanModel_S2V DiT weights from 28 GB to ~14 GB.',
                        )
                    with gr.Row():
                        btn_load = gr.Button('Pre-load Local Engine')
                        btn_local = gr.Button('Generate (Local)', variant='primary')
                with gr.Column(scale=3):
                    vid_out_local = gr.Video(label='Generated video', height=480)
                    proc_img_local = gr.Image(label='Reference image (echo)', height=120, interactive=False)
                    status_local = gr.Markdown('')
            fast_lora_local.change(on_fast_lora_toggle, inputs=[fast_lora_local], outputs=[steps_local, cfg_local])
            btn_load.click(local_load, inputs=[quant_local, fast_lora_local], outputs=status_local)
            btn_local.click(
                do_local,
                inputs=[img_local, aud_local, prompt_local, res_local, pose_local, frames_local, steps_local, cfg_local, seed_local, quant_local, fast_lora_local],
                outputs=[vid_out_local, proc_img_local, status_local],
            )

        # -- Cloud API tab --
        with gr.Tab('🌐 Cloud API (DashScope)'):
            with gr.Row():
                with gr.Column(scale=2):
                    img_cloud = gr.Image(
                        label='Character image (the subject to lip-sync)',
                        type='filepath', image_mode='RGB', height=280,
                    )
                    aud_cloud = gr.Audio(
                        label='Audio clip (the speech or song to lip-sync to)',
                        type='filepath', sources=['upload'],
                        info='WAV / MP3. The audio length determines the output video length. Speech or singing.',
                    )
                    prompt_cloud = gr.Textbox(
                        label='Prompt (optional, advanced)',
                        value='',
                        placeholder='e.g. "A man with short gray hair plays a red electric guitar."',
                        info='Describes the scene. If empty, the model uses the image and audio alone.',
                    )
                    with gr.Row():
                        res_cloud = gr.Dropdown(
                            choices=RESOLUTIONS, value=DEFAULT_RESOLUTION, label='Resolution',
                            info='480P = faster + cheaper. 720P = higher quality.',
                        )
                    btn_cloud = gr.Button('Generate (Cloud API)', variant='primary')
                with gr.Column(scale=3):
                    vid_out_cloud = gr.Video(label='Generated video', height=480)
                    proc_img = gr.Image(label='Reference image (echo)', height=120, interactive=False)
                    status_cloud = gr.Markdown('')
            btn_cloud.click(
                do_cloud,
                inputs=[img_cloud, aud_cloud, prompt_cloud, res_cloud],
                outputs=[vid_out_cloud, proc_img, status_cloud],
            )

        # -- VRAM / status tab --
        with gr.Tab('VRAM'):
            vram_text = gr.Textbox(label='Status', value=vram_status, interactive=False, lines=5, info='Current GPU memory allocation and local S2V status.')
            with gr.Row():
                gr.Button('Free local engine').click(
                    lambda: (LOCAL_ENGINE.unload(), vram_status())[1],
                    outputs=vram_text,
                )
                gr.Button('Refresh status').click(vram_status, outputs=vram_text)

clear_output()
print('[UI] Launching ...')
demo.queue(max_size=4, concurrency_limit=2)
demo.load(
    lambda: 'Wan 2.2 S2V ready — use the Local tab for single-GPU inference (L4/A100) or the Cloud tab with DASHSCOPE_API_KEY.',
    outputs=[status_local],
)
demo.launch(share=True, show_error=True, height=860, prevent_thread_lock=False)
print('\n[UI] Public URL above. Open it in a new tab.')

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings a tiny URL every 60 s to keep the Colab tab active.

import time, requests, datetime, threading, IPython
_stop = threading.Event()
def _pinger():
    while not _stop.is_set():
        try:
            requests.get('https://huggingface.co/api/models/Wan-AI/Wan2.2-S2V-14B', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} OK. Stop with `_stop.set()`.')
        except Exception as e:
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} WARN: {e}')
        _stop.wait(60)
_t = threading.Thread(target=_pinger, daemon=True)
_t.start()
print('[Keep-Alive] Started.')

In [ ]:
# @title STEP 6 — Quick Test (Local Single-GPU or Cloud API, no UI)
# @markdown Runs a single 480P S2V generation job using either the **Local Single-GPU** engine
# @markdown or the **Cloud API**. If `QUICK_IMAGE` or `QUICK_AUDIO` are missing, synthetic test files
# @markdown are created automatically so you can smoke-test the pipeline immediately.

import os, time, traceback, pathlib, shutil, wave, struct, math
from PIL import Image, ImageDraw
from IPython.display import display, FileLink

QUICK_BACKEND = 'local'  # @param ['local', 'cloud']
QUICK_RESOLUTION = '480P'  # @param ['480P', '720P']
QUICK_IMAGE = '/content/character.png'  # @param {type:"string"}
QUICK_AUDIO = '/content/speech.wav'  # @param {type:"string"}

if not os.path.exists(QUICK_IMAGE):
    img = Image.new('RGB', (480, 832), color=(32, 42, 64))
    draw = ImageDraw.Draw(img)
    draw.ellipse((160, 180, 320, 360), fill=(240, 200, 175))
    draw.rectangle((130, 365, 350, 720), fill=(75, 135, 210))
    img.save(QUICK_IMAGE)
    print(f'[QuickTest] Created synthetic portrait image at {QUICK_IMAGE}')

if not os.path.exists(QUICK_AUDIO):
    sr = 16000
    dur = 2.0
    with wave.open(QUICK_AUDIO, 'w') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        for i in range(int(sr * dur)):
            env = 0.5 * (1.0 - math.cos(2 * math.pi * i / (sr * 0.4)))
            val = int(12000 * env * math.sin(2 * math.pi * 220.0 * i / sr))
            wf.writeframesraw(struct.pack('<h', val))
    print(f'[QuickTest] Created synthetic 2s audio clip at {QUICK_AUDIO}')

t0 = time.time()
try:
    if QUICK_BACKEND == 'cloud':
        if not CLOUD.api_key:
            raise SystemExit('DASHSCOPE_API_KEY is not set. Switch QUICK_BACKEND to "local" or set DASHSCOPE_API_KEY in Colab secrets.')
        print(f'[QuickTest] Cloud API at {QUICK_RESOLUTION} ...')
        out_path = CLOUD.submit(
            image_path=QUICK_IMAGE,
            audio_path=QUICK_AUDIO,
            resolution=QUICK_RESOLUTION,
        )
    else:
        print(f'[QuickTest] Local Single-GPU S2V at {QUICK_RESOLUTION} (INT8={DEFAULT_QUANTIZE_INT8}, 4-Step={DEFAULT_FAST_4STEP_LORA}) ...')
        out_path = LOCAL_ENGINE.generate(
            image_path=QUICK_IMAGE,
            audio_path=QUICK_AUDIO,
            resolution=QUICK_RESOLUTION,
            infer_frames=32,
            quantize_int8=DEFAULT_QUANTIZE_INT8,
            use_fast_lora=DEFAULT_FAST_4STEP_LORA,
        )
    final = DRIVE_OUT / f'quicktest_{int(time.time())}.mp4'
    shutil.copy2(out_path, str(final))
    print(f'[QuickTest] DONE in {time.time() - t0:.1f}s -> {final}')
    display(FileLink(str(final)))
except Exception as e:
    print(f'[QuickTest] FAILED: {e}')
    traceback.print_exc()
    raise

In [ ]:
# @title STEP 7 — Batch Generation (Local Single-GPU or Cloud API)
# @markdown Reads a folder where each subfolder contains an `image.{png,jpg,jpeg,webp}` and
# @markdown an `audio.{wav,mp3}` pair. Runs either the **Local Single-GPU** engine or the **Cloud API**
# @markdown per pair with per-item `try`/`except`. Outputs are saved to `DRIVE_OUT`.

import os, time, traceback, pathlib, shutil
from tqdm import tqdm
from IPython.display import display, FileLink

# --- Edit these before running ---------------------------------
BATCH_BACKEND = 'local'  # @param ['local', 'cloud']
BATCH_RESOLUTION = '480P'  # @param ['480P', '720P']
BATCH_INPUT_DIR = '/content/wan22_s2v_pairs'  # @param {type:"string"}
MAX_ITEMS = 0  # @param {type:"integer"}
# --------------------------------------------------------------

if BATCH_BACKEND == 'cloud' and not CLOUD.api_key:
    raise SystemExit('DASHSCOPE_API_KEY is not set. Switch BATCH_BACKEND to "local" or set DASHSCOPE_API_KEY.')

INPUT_DIR = pathlib.Path(BATCH_INPUT_DIR)
if not INPUT_DIR.is_dir():
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    sample = INPUT_DIR / 'example_pair'
    sample.mkdir(exist_ok=True)
    if os.path.exists('/content/character.png') and os.path.exists('/content/speech.wav'):
        shutil.copy2('/content/character.png', str(sample / 'image.png'))
        shutil.copy2('/content/speech.wav', str(sample / 'audio.wav'))
        print(f'[Batch] Bootstrapped {sample} with sample character.png + speech.wav.')
    else:
        print(f'[Batch] Created {sample}; place an image and audio file inside each pair folder and re-run.')
        raise SystemExit('Input dir was empty; created example_pair directory.')

pairs = []
img_exts = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
aud_exts = {'.wav', '.mp3'}
for sub in sorted(p for p in INPUT_DIR.iterdir() if p.is_dir()):
    imgs = [p for p in sub.iterdir() if p.suffix.lower() in img_exts]
    auds = [p for p in sub.iterdir() if p.suffix.lower() in aud_exts]
    if imgs and auds:
        pairs.append((sub.name, imgs[0], auds[0]))
if MAX_ITEMS:
    pairs = pairs[:MAX_ITEMS]
print(f'[Batch] {len(pairs)} pair(s) queued — Backend: {BATCH_BACKEND} | Resolution: {BATCH_RESOLUTION}')

ok = 0; fail = 0
for i, (name, img, aud) in enumerate(tqdm(pairs, desc='Wan2.2 S2V Batch'), 1):
    try:
        t0 = time.time()
        if BATCH_BACKEND == 'cloud':
            out_path = CLOUD.submit(
                image_path=str(img), audio_path=str(aud),
                resolution=BATCH_RESOLUTION,
            )
        else:
            out_path = LOCAL_ENGINE.generate(
                image_path=str(img), audio_path=str(aud),
                resolution=BATCH_RESOLUTION,
                quantize_int8=DEFAULT_QUANTIZE_INT8,
                use_fast_lora=DEFAULT_FAST_4STEP_LORA,
            )
        safe = ''.join(c if c.isalnum() else '_' for c in name[:30]).strip('_') or f'item_{i:02d}'
        final = DRIVE_OUT / f'{i:03d}_{safe}.mp4'
        shutil.copy2(out_path, str(final))
        print(f'  [{i:03d}] {name} -> {final.name} ({time.time() - t0:.1f}s)')
        display(FileLink(str(final)))
        ok += 1
    except Exception as e:
        print(f'  [{i:03d}] EXCEPTION: {e}'); traceback.print_exc(); fail += 1
print(f'\n[Batch] DONE: {ok} OK / {fail} failed / {len(pairs)} total -> {DRIVE_OUT}')